# 02 · Model training

Fine-tunes YOLOv8 on the prepared dataset with checkpoints on Drive and **automatic resume** if the runtime disconnects. Re-running the training cell always does the right thing.

> **Runtime:** go to *Runtime → Change runtime type → T4 GPU* before running anything.
> Every cell below is safe to re-run; nothing is lost when Colab disconnects because all
> data, checkpoints and results live on your Google Drive.

## 0.1 GPU check
**What:** prints the GPU Colab assigned to this session.
**Why:** training on CPU takes hours instead of minutes; we warn loudly if no GPU is present.

In [ ]:
import subprocess, shutil

try:
    if shutil.which("nvidia-smi") is None:
        raise FileNotFoundError("nvidia-smi not found")
    print(subprocess.check_output(["nvidia-smi"], encoding="utf-8", errors="replace"))
    GPU_AVAILABLE = True
except Exception as exc:
    GPU_AVAILABLE = False
    print("=" * 70)
    print("WARNING: No GPU detected (", exc, ")")
    print("Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run.")
    print("=" * 70)

## 0.2 Mount Google Drive & get the code
**What:** mounts Drive at `/content/drive`, then either uses a copy of the repo already on Drive
or clones it from GitHub into `/content`.
**Why:** Drive is the only storage that survives a runtime disconnect. Checkpoints, the prepared
dataset and evaluation outputs are all written under `SAVE_DIR`.

Edit `REPO_URL` once (your fork), or copy the repo folder to
`MyDrive/masked-face-detection/repo` and it will be picked up automatically.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Numbu-bit/masked-face-detection.git"   # <-- change if you fork
SAVE_DIR = "/content/drive/MyDrive/masked-face-detection"                  # everything persistent lives here
DRIVE_REPO = os.path.join(SAVE_DIR, "repo")
LOCAL_REPO = "/content/masked-face-detection"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab - using the current working directory as the repo.")
    SAVE_DIR = os.path.abspath("runs")

os.makedirs(SAVE_DIR, exist_ok=True)

if IN_COLAB:
    if os.path.isfile(os.path.join(DRIVE_REPO, "configs", "default.yaml")):
        REPO_DIR = DRIVE_REPO
        print("Using repo copy on Drive:", REPO_DIR)
    else:
        REPO_DIR = LOCAL_REPO
        if not os.path.isfile(os.path.join(REPO_DIR, "configs", "default.yaml")):
            if "YOUR_USERNAME" in REPO_URL:
                raise RuntimeError(
                    "Set REPO_URL to your GitHub fork above, OR copy the repository folder to "
                    f"{DRIVE_REPO} so the notebook can find configs/default.yaml.")
            rc = os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
            if rc != 0:
                raise RuntimeError(f"git clone failed for {REPO_URL}. Is the repo public / URL correct?")
        else:
            os.system(f"git -C {REPO_DIR} pull -q")
else:
    REPO_DIR = os.getcwd() if os.path.isfile("configs/default.yaml") else os.path.abspath("..")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)
print("Persistent storage:", SAVE_DIR)

## 0.3 Install dependencies
**What:** installs the tested stack from `requirements.txt`. If that cannot be installed on this
runtime's Python version, it automatically falls back to `requirements-fallback.txt`
(same libraries, range pins) and tells you so.
**Why:** exact pins avoid the "it worked yesterday" class of breakages, but Colab upgrades its
Python from time to time and old pins may have no wheels for it — the fallback keeps you running.
Takes ~1–2 minutes on a fresh runtime; instant on re-runs.

In [ ]:
import subprocess, sys, platform

print("Python", platform.python_version())


def pip_install(req_file: str) -> "subprocess.CompletedProcess":
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file],
                          capture_output=True, encoding="utf-8", errors="replace")


proc = pip_install("requirements.txt")
if proc.returncode == 0:
    print("Dependencies installed (requirements.txt).")
else:
    print("requirements.txt could not be installed on this runtime. pip said:\n")
    print(proc.stderr[-2500:])
    print("\n-> Trying requirements-fallback.txt (range pins) ...")
    proc = pip_install("requirements-fallback.txt")
    if proc.returncode == 0:
        print("Dependencies installed (requirements-fallback.txt).")
    else:
        print(proc.stderr[-2500:])
        raise RuntimeError("Both requirement sets failed. Copy the pip output above into an issue / to Claude.")

import ultralytics, torch, numpy, cv2
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| numpy", numpy.__version__,
      "| opencv", cv2.__version__, "| CUDA", torch.cuda.is_available())

## 0.4 Seeds & config
**What:** loads `configs/default.yaml` (the single source of truth for every hyper-parameter)
and seeds Python / NumPy / PyTorch / CUDA with `seed=42`.
**Why:** reproducible splits, reproducible training.

In [ ]:
import json
from src.utils import load_config, set_seed, get_device, resolve_save_dir

# Optional: MFD_OVERRIDES='{"epochs": 1}' in the environment (used by automated tests)
ENV_OVERRIDES = json.loads(os.environ.get("MFD_OVERRIDES", "{}"))
cfg = load_config(overrides={"save_dir": os.path.join(SAVE_DIR, "runs"), **ENV_OVERRIDES})
set_seed(cfg["seed"])
device = get_device()
RUN_DIR = resolve_save_dir(cfg) / cfg["run_name"]
DATA_ROOT = cfg["data_root"]
print("Model:", cfg["model_variant"], "| image size:", cfg["image_size"], "| batch:", cfg["batch_size"])
print("Run directory:", RUN_DIR)

## Restore the prepared dataset
**What:** if `/content/data` is missing (new runtime), unzips the copy that notebook 01 saved to Drive.
**Why:** `/content` is wiped on every disconnect; Drive is not.

In [ ]:
import zipfile
from pathlib import Path

DATA_ZIP = Path(SAVE_DIR) / "data_prepared.zip"
DATA_YAML = Path(DATA_ROOT) / "data.yaml"

if not DATA_YAML.exists():
    if DATA_ZIP.exists():
        print("Restoring dataset from", DATA_ZIP, "...")
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(Path(DATA_ROOT).parent)
        print("Restored.")
    else:
        raise FileNotFoundError(
            f"Neither {DATA_YAML} nor {DATA_ZIP} exists. Run notebook 01_data_preparation.ipynb first.")

from src.dataset import dataset_statistics
import pandas as pd
display(pd.DataFrame(dataset_statistics(Path(DATA_ROOT), cfg["class_names"])).T)

## Keep the session alive (optional)
**What:** clicks Colab's *connect* button every 60 s from JavaScript.
**Why:** Colab disconnects idle tabs after ~30 minutes; long training runs would die.
Run this once and leave the tab open.

In [ ]:
%%javascript
function KeepAlive() {
  const btn = document.querySelector("colab-connect-button");
  if (btn) { btn.click(); }
}
setInterval(KeepAlive, 60000);
console.log("Keep-alive installed.");

## 1. Choose the model size
**What:** optional overrides of the config for this session.
**Why:** free-tier T4 (15 GB VRAM) handles `yolov8s @ 640, batch 16` comfortably (~2 min/epoch on
~600 images). If you hit *CUDA out of memory*, drop `batch_size` to 8 or `image_size` to 416.

| Tier | Suggested | Time for 100 epochs (Kaggle set) |
|---|---|---|
| Free (T4) | `yolov8n` or `yolov8s`, batch 16 | ~1–3 h |
| **Pro (A100/L4) — optional** | `yolov8m`, batch 32 | ~1 h |

In [ ]:
# ---- Session overrides (leave as-is to use configs/default.yaml) ----------
OVERRIDES = {
    # "model_variant": "yolov8n",   # faster; use on free tier if time is short
    # "batch_size": 8,              # if you see CUDA OOM
    # "image_size": 416,            # if you see CUDA OOM
    # "epochs": 50,                 # quicker experiment
}
# ---- Pro-tier only (OPTIONAL) - uncomment on an A100 / L4 runtime ----------
# OVERRIDES.update({"model_variant": "yolov8m", "batch_size": 32})

cfg = load_config(overrides={"save_dir": os.path.join(SAVE_DIR, "runs"), **ENV_OVERRIDES, **OVERRIDES})
if OVERRIDES.get("model_variant"):
    cfg["run_name"] = f"{cfg['model_variant']}_mask"
RUN_DIR = resolve_save_dir(cfg) / cfg["run_name"]
print({k: cfg[k] for k in ("model_variant", "image_size", "batch_size", "epochs", "patience", "run_name")})
print("Checkpoints ->", RUN_DIR / "weights")

## 2. Smoke test (1 epoch) — optional but recommended
**What:** trains for a single epoch into a throw-away run folder.
**Why:** catches dataset / path / memory problems in 2 minutes instead of discovering them 40
minutes into the real run.

In [ ]:
from src.train import train_yolo
from src.utils import free_memory
import shutil

smoke_cfg = dict(cfg, run_name="smoke_test", epochs=1, patience=1)
shutil.rmtree(resolve_save_dir(smoke_cfg) / "smoke_test", ignore_errors=True)
try:
    best = train_yolo(smoke_cfg, str(DATA_YAML), resume=False)
    print("Smoke test OK ->", best)
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        raise RuntimeError("CUDA OOM: set OVERRIDES['batch_size']=8 (or image_size=416) in the cell above.") from exc
    raise
finally:
    free_memory()

## 3. Train (auto-resumes after a disconnect)
**What:** fine-tunes from COCO weights with the settings in `configs/default.yaml`
(AdamW, cosine LR, 3 warm-up epochs, mosaic + mixup, early stopping `patience=15`,
checkpoint every 10 epochs + `last.pt` every epoch).
**Why it is safe to re-run:** `train_yolo` checks for `RUN_DIR/weights/last.pt`; if it exists
training continues from that epoch with the original arguments. If the run already finished it
tells you so instead of starting over.

Progress is printed per epoch by Ultralytics; `results.csv` and plots are written to `RUN_DIR`.

In [ ]:
from src.train import train_yolo
from src.utils import find_last_checkpoint, free_memory
import time

last = find_last_checkpoint(cfg)
print("Resuming from", last if last else "scratch (no last.pt found)")

t0 = time.time()
try:
    BEST_PT = train_yolo(cfg, str(DATA_YAML))        # resume=None -> auto-detect
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        raise RuntimeError("CUDA OOM - lower batch_size / image_size in OVERRIDES and re-run; "
                           "the run will resume from last.pt.") from exc
    raise
finally:
    free_memory()
print(f"Done in {(time.time() - t0) / 60:.1f} min. Best weights: {BEST_PT}")

## 4. Training curves & quick validation
**What:** shows Ultralytics' `results.png` and runs validation on the **val** split with `best.pt`.
**Why:** a first look before the full evaluation in notebook 03.

In [ ]:
from IPython.display import Image, display
from src.model import load_trained_yolo
from src.evaluate import plot_training_curves
import matplotlib.pyplot as plt

fig = plot_training_curves(cfg); plt.show()
for name in ("results.png", "confusion_matrix.png"):
    p = RUN_DIR / name
    if p.exists():
        display(Image(filename=str(p), width=900))

model = load_trained_yolo(cfg)
metrics = model.val(data=str(DATA_YAML), split="val", imgsz=cfg["image_size"], plots=False, verbose=False)
print(f"val mAP@0.5 = {metrics.box.map50:.4f} | mAP@0.5:0.95 = {metrics.box.map:.4f}")
free_memory()

## 5. Free VRAM
**What:** clears CUDA cache and Python garbage.
**Why:** keeps the runtime healthy if you go straight to notebook 03 in the same session.
Best weights are at `RUN_DIR/weights/best.pt` on Drive — continue with `03_evaluation.ipynb`.

In [ ]:
import torch, gc
del model
gc.collect(); torch.cuda.empty_cache()
print("VRAM freed. Best checkpoint:", BEST_PT)